# Calibration of the radio materials with neural materials

**Purpose.** This notebook calibrates the radio materials of the Sionna RT scene against the
minimization-of-drive-tests (MDT) records. A neural network gives the material properties at
each point of the scene. Gradient descent changes the network until the simulated reference
signal received power (RSRP) agrees with the measured RSRP. The method is "Neural Materials"
of Hoydis et al., *Learning Radio Environments by Differentiable Ray Tracing*,
[arXiv:2311.18558](https://arxiv.org/abs/2311.18558).

**Inputs.**

| File | Content |
|---|---|
| `configs/config.yaml` | The configuration. `src.utils.load_config` composes it with the other files of `configs/`. The notebook reads `seed`, the Sionna RT configuration (`simulation`), and the configuration of the calibration (`calibration`: the network, the optimizer, and the number of steps). |
| `data/raw/cells.csv` | One row for each cell |
| `data/processed/train.parquet` | The train records. The calibration calculates the gradients from them. |
| `data/processed/validation.parquet` | The validation records. They show the loss during the calibration. |
| `data/processed/test.parquet` | The test records. They give the error after the calibration. |

**Method.** `src.simulation.NeuralRadioMaterial` replaces the materials of the scene.
`src.calibration.calibrate` does the gradient steps. Section 2 gives the equations.

**Figures.** `configure_matplotlib` sets the figure style. Each figure is saved as a PNG file
in `reports/figures/`. The text below each figure is its caption.

**Outputs.**

| File | Content |
|---|---|
| `outputs/calibration/calibration.npz` | The weights of the network, the network structure, the normalization of the positions, and the power scale |
| `outputs/calibration/history.csv` | The loss and the power scale at each step |

**Requirements.** Run `task scenario` and `notebooks/02_preprocessing.ipynb` first. The
notebook needs an NVIDIA GPU. The run time and the GPU memory increase with `calibration.batch_size`,
with the size of the network, and with `simulation.solver.samples_per_src`.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm
from omegaconf import OmegaConf

from src.calibration import (
    absolute_log_error,
    calibrate,
    dbm_to_mw,
    save_calibration,
    scale_estimate,
)
from src.core import read_cells
from src.data import SPLIT_NAMES, mdt_to_arrays
from src.simulation import (
    apply_neural_material,
    building_height_map,
    compute_rsrp,
    load_rt_scene,
    make_material_network,
)
from src.utils import (
    COLORS,
    DATA_DIR,
    FIGURE_WIDTH_DOUBLE,
    OUTPUTS_DIR,
    PROJECT_ROOT,
    load_config,
    save_figure,
    save_table,
    setup_notebook,
)

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
CALIBRATION_DIR = OUTPUTS_DIR / "calibration"
cfg = load_config()
device = setup_notebook(cfg)

In [ ]:
def label_panels(axes):
    "Put the labels (a), (b), ... above the top left corner of each panel."
    for index, ax in enumerate(np.ravel(axes)):
        ax.set_title(f"({chr(ord('a') + index)})", loc="left", fontweight="bold")


def format_map(ax):
    "Give a map panel an equal scale on the two axes and axis labels in meters."
    ax.set(xlabel="$x$ (m)", ylabel="$y$ (m)", aspect="equal")
    ax.grid(False)


def ecdf(values):
    "Return the sorted values and the empirical cumulative distribution function."
    ordered = np.sort(np.asarray(values))
    return ordered, np.arange(1, ordered.size + 1) / ordered.size


def received_powers(measured_dbm, simulated_dbm):
    "Return the measured and the simulated powers (mW) of the received (record, cell) pairs."
    received = np.isfinite(measured_dbm)
    # No path gives NaN in dBm. It is a power of 0.
    return dbm_to_mw(measured_dbm[received]), np.nan_to_num(dbm_to_mw(simulated_dbm[received]))

The configuration of the calibration is below.

In [ ]:
print(OmegaConf.to_yaml(cfg.calibration))

## 2. Method

### 2.1 Material properties and the ray tracer

Sionna RT calculates each propagation path from a cell to a UE. At each point where a path
hits a surface, the field changes. The change is a function of the material at that point.
A material has the relative permittivity $\varepsilon_r \geq 1$, the conductivity
$\sigma \geq 0$ (S/m), the scattering coefficient $S \in [0, 1]$, and the cross-polarization
discrimination (XPD) coefficient $K_x \in [0, 1]$. The complex relative permittivity is
(arXiv:2311.18558, Eq. 15)

$$\eta = \varepsilon_r - j\,\frac{\sigma}{\varepsilon_0\,\omega},$$

where $\varepsilon_0$ is the vacuum permittivity and $\omega = 2\pi f$ is the angular
frequency. $\eta$ sets the reflection coefficients of a surface. $S^2$ is the fraction of the
reflected energy that goes to diffuse reflections. $K_x$ is the fraction of the diffuse energy
that goes to the orthogonal polarization.

Sionna RT models a wall as a slab with a thickness $d$. The thickness changes the reflected
and the transmitted field. The paper has no thickness, because its ray tracer ignores
refraction.

Each step of the field calculation is differentiable for these properties. The ray tracer is
thus a computational graph, and the material properties are its trainable parameters.

### 2.2 Neural materials

A scene file gives one material to each object. The paper calls the calibration of these
values "Learned Materials". "Neural Materials" (arXiv:2311.18558, Section III-D) do not use the
division into objects. One multilayer perceptron (MLP) gives the material properties at each
position $\mathbf{p} = [x, y, z]^\mathsf{T}$ of the scene:

1. **Normalization** (Eq. 37). Let $\mathbf{b}$ be the center of the bounding box of the scene
   and let $\Delta_b$ be the length of its longest side. The normalized position is
   $\bar{\mathbf{p}} = (\mathbf{p} - \mathbf{b}) / \Delta_b$.
2. **Positional encoding** (Eq. 36). Each coordinate $p$ of $\bar{\mathbf{p}}$ becomes
   $\gamma(p) = [\sin(2\pi p), \cos(2\pi p), \ldots, \sin(2^{L}\pi p), \cos(2^{L}\pi p)]$.
   $L$ is `calibration.network.encoding_octaves`. A larger $L$ lets the properties change faster in space.
3. **MLP.** The network has `calibration.network.hidden_layers` hidden layers of `calibration.network.hidden_units`
   neurons with ReLU activations. It has one output $y_i$ for each property.
4. **Output transformations** (Fig. 1 of the paper, with the functions of Eq. 32 to 35). They
   keep each property in its valid range:

$$\varepsilon_r = 1 + e^{y_1}, \qquad \sigma = e^{y_2}, \qquad
S = \operatorname{sigmoid}(y_3), \qquad K_x = \operatorname{sigmoid}(y_4), \qquad d = e^{y_5}.$$

`NeuralRadioMaterial` is a subclass of the Sionna RT `RadioMaterial`. It calculates the five
properties at the point of each ray interaction, then uses the field calculations of its
parent class. All objects of the scene get the same `NeuralRadioMaterial`.

### 2.3 Loss

The loss of the paper (Eq. 38) is the sum of two terms: one for the delay spread and one for
the power. An MDT record has only the RSRP of each cell, thus the delay spread has no measured
value. This notebook uses only the power term. For one received (record, cell) pair, let $P$
be the measured RSRP and let $\hat{P}$ be the simulated RSRP, the two in milliwatts. The loss
of the pair is the symmetric absolute percentage error (Eq. 39)

$$l(\alpha P, \hat{P}) = \frac{|\alpha P - \hat{P}|}{\alpha P + \hat{P}} \in [0, 1].$$

The loss of a batch is the mean of $l$ for all received pairs of the batch. The normalization
gives the same weight to a weak pair and to a strong pair. A pair with no simulated path has
$\hat{P} = 0$ and a loss of 1.

### 2.4 Power scale

The measured powers can have a common offset from the true powers. If the calibration ignores
the offset, the materials must compensate for it, and their values become incorrect. The paper
thus multiplies the measured power by a scale $\alpha$ (Section III-F). For a batch of $B$
pairs, the least-squares scale is (Eq. 43)

$$\hat{\alpha}_i = \arg\min_\alpha \frac{1}{B}\sum_{b=1}^{B}\left(\alpha P_b - \hat{P}_b\right)^2
= \frac{\sum_b P_b \hat{P}_b}{\sum_b P_b^2}.$$

An exponential moving average with the decay $\delta$ (`scale_decay`) removes the variation
between batches (Eq. 42):

$$\alpha_i = \delta\,\alpha_{i-1} + (1 - \delta)\,\hat{\alpha}_i, \qquad \alpha_0 = \hat{\alpha}_1.$$

$\alpha$ gets no gradient. The sums of Eq. 43 are in linear power, thus the strongest pairs of
a batch control $\hat{\alpha}_i$.

### 2.5 Training algorithm

`calibrate` does Algorithm 1 of the paper. Each step uses a random batch of `batch_size` train
records.

```mermaid
flowchart LR
    net[make_material_network] --> mat[NeuralRadioMaterial]
    mat -- apply_neural_material --> scene[Sionna RT scene]
    scene --> rsrp[compute_rsrp_mw]
    rsrp --> scale[scale_estimate and update_scale]
    scale --> loss[smape]
    rsrp --> loss
    loss -- gradient --> opt[AdamW step]
    opt -- new weights --> net
```

1. Ray-trace the batch: `compute_rsrp_mw` gives $\hat{P}$ with gradients.
2. Calculate $\hat{\alpha}_i$ and $\alpha_i$.
3. Calculate the mean loss and its gradient for the weights of the MLP.
4. Do one step of the AdamW optimizer.

### 2.6 Differences from the paper

| Topic | Paper | This notebook | Cause |
|---|---|---|---|
| Loss | Delay spread term and power term | Power term only | An MDT record has no delay spread. |
| Measured quantity | Total channel gain of a channel impulse response | RSRP of each (record, cell) pair | Format of the MDT data |
| Thickness | Not a parameter | Fifth output of the MLP | Sionna RT models a wall as a slab. The exponential function for $d$ is an assumption. |
| Optimizer | Adam | AdamW | AdamW with no weight decay is Adam. |
| Diffuse reflections | Off for the measured data | On | The calibration learns $S$ and $K_x$. |
| Weights of the MLP | Not specified | Half precision | The Dr.Jit CUDA backend evaluates the MLP only in half precision. |

## 3. Data

`mdt_to_arrays` changes each part to one position for each record and one RSRP for each
(record, cell) pair. A missing RSRP shows that the record does not have the cell. Table 1 gives
the size of the parts.

In [ ]:
cells = read_cells(RAW_DIR / "cells.csv")
frames = {name: pd.read_parquet(PROCESSED_DIR / f"{name}.parquet") for name in SPLIT_NAMES}
positions, rsrp_dbm = {}, {}
for name, frame in frames.items():
    positions[name], rsrp_dbm[name] = mdt_to_arrays(frame, cells)

save_table(
    pd.DataFrame(
        {
            "Records": [len(positions[name]) for name in SPLIT_NAMES],
            "Received (record, cell) pairs": [len(frames[name]) for name in SPLIT_NAMES],
            "Median RSRP (dBm)": [frames[name]["rsrp_dbm"].median() for name in SPLIT_NAMES],
        },
        index=list(SPLIT_NAMES),
    )
    .style.set_caption("Table 1. Size of the parts of the MDT data.")
    .format({"Median RSRP (dBm)": "{:.1f}"}),
    "calibration_parts",
)

The scene and its height map give the background of the maps in this notebook.

In [ ]:
scene = load_rt_scene(cfg.simulation.scene.name)
height_map = building_height_map(scene, cfg.scenario.grid.tile_size_m)
heights = height_map.heights
ny, nx = heights.shape
map_extent = [
    height_map.x_min,
    height_map.x_min + nx * height_map.tile_size,
    height_map.y_min,
    height_map.y_min + ny * height_map.tile_size,
]
is_building = heights > cfg.scenario.grid.min_building_height_m

fig, (ax_map, ax_cdf) = plt.subplots(
    1, 2, figsize=(FIGURE_WIDTH_DOUBLE, 2.9), width_ratios=[1.5, 1]
)
ax_map.imshow(is_building, origin="lower", extent=map_extent, cmap="Greys", alpha=0.3)
for color, name in zip(COLORS, SPLIT_NAMES, strict=False):
    ax_map.scatter(*positions[name][:, :2].T, s=3, color=color, linewidths=0, label=name)
    ax_cdf.plot(*ecdf(frames[name]["rsrp_dbm"]), color=color, label=name)
ax_map.scatter(
    [cell.x for cell in cells], [cell.y for cell in cells], marker="^", color="black", label="cell"
)
format_map(ax_map)
ax_map.legend(loc="upper right", markerscale=3)
ax_cdf.set(xlabel="RSRP (dBm)", ylabel="Empirical CDF", ylim=(0, 1))
ax_cdf.legend(loc="upper left")
label_panels([ax_map, ax_cdf])
save_figure(fig, "calibration_data")
plt.show()

**Fig. 1.** Parts of the MDT data. (a) Reported positions of the records of each part. The
gray areas are buildings and the triangles are the sites. (b) Empirical distribution of the
reported RSRP of each part.

## 4. Baseline: ITU materials

The scene file gives each object a material of Recommendation ITU-R P.2040. The paper uses
this configuration as the baseline "ITU Materials". The scenario generator used the same
materials to make the MDT records, thus the baseline is also the model that made the data.
Only the measurement errors of the generator make a difference between the baseline and the
records.

The baseline gets its own power scale from Eq. 43 with all train records. Table 2 gives the
materials of the scene at the carrier frequency.

In [ ]:
itu_dbm = {
    name: compute_rsrp(scene, cells, positions[name], cfg.simulation) for name in ("train", "test")
}
itu_scale = scale_estimate(*received_powers(rsrp_dbm["train"], itu_dbm["train"]))

objects_of = pd.Series(
    [scene_object.radio_material.name for scene_object in scene.objects.values()]
).value_counts()
save_table(
    pd.DataFrame(
        [
            {
                "Material": name,
                "Objects": int(objects_of.get(name, 0)),
                "Relative permittivity": float(material.relative_permittivity.numpy()[0]),
                "Conductivity (S/m)": float(material.conductivity.numpy()[0]),
                "Thickness (m)": float(material.thickness.numpy()[0]),
                "Scattering coefficient": float(material.scattering_coefficient.numpy()[0]),
            }
            for name, material in scene.radio_materials.items()
        ]
    )
    .style.set_caption("Table 2. ITU materials of the scene at the carrier frequency.")
    .format({"Conductivity (S/m)": "{:.3g}"}, precision=3)
    .hide(axis="index"),
    "calibration_itu_materials",
)

## 5. Calibration

`make_material_network` makes the MLP with random initial weights. `apply_neural_material`
gives one `NeuralRadioMaterial` to all objects of the scene. `calibrate` then does the gradient
steps. At the first step, the material properties are thus arbitrary.

In [ ]:
network = make_material_network(**cfg.calibration.network, seed=cfg.seed)
material = apply_neural_material(scene, network)
result = calibrate(
    scene,
    cells,
    network,
    positions["train"],
    rsrp_dbm["train"],
    (positions["validation"], rsrp_dbm["validation"]),
    cfg.calibration,
    cfg.simulation,
    seed=cfg.seed,
)
history = result.history

The notebook writes the result to `outputs/calibration/`. The weights and the power scale are
in one file, because the calibration learns them together: the weights are correct only with
their scale. `src.calibration.load_calibration` reads the file and gives the material and the
scale. Table 3 gives the files.

In [ ]:
CALIBRATION_DIR.mkdir(parents=True, exist_ok=True)
save_calibration(CALIBRATION_DIR / "calibration.npz", material, result.scale)
history.to_csv(CALIBRATION_DIR / "history.csv", index=False)

save_table(
    pd.DataFrame(
        [
            {
                "File": path.relative_to(PROJECT_ROOT).as_posix(),
                "Size (kB)": path.stat().st_size / 1e3,
            }
            for path in sorted(CALIBRATION_DIR.iterdir())
        ]
    )
    .style.set_caption("Table 3. Files in outputs/calibration.")
    .format({"Size (kB)": "{:.1f}"})
    .hide(axis="index"),
    "calibration_files",
)

## 6. Learning curves

Fig. 2 shows the loss and the power scale at each step. The loss of one step is the mean for
one batch, thus it has much variation. The validation loss is the mean for all validation
records.

The scale has an expected value. The generator adds a noise with the mean
`scenario.measurement.rsrp_noise_mean_db` to each true RSRP. If the materials are correct, the scale
compensates for this mean: $10 \log_{10} \alpha$ is then the negative of the mean.

In [ ]:
validated = history.dropna(subset="validation_loss")
window = max(len(history) // 50, 1)

fig, (ax_loss, ax_scale) = plt.subplots(1, 2, figsize=(FIGURE_WIDTH_DOUBLE, 2.6))
ax_loss.plot(history["step"], history["loss"], color=COLORS[5], linewidth=0.5, label="Batch")
ax_loss.plot(
    history["step"],
    history["loss"].rolling(window, min_periods=1).mean(),
    color=COLORS[0],
    label="Batch, moving mean",
)
ax_loss.plot(
    validated["step"],
    validated["validation_loss"],
    "o-",
    color=COLORS[1],
    markersize=2,
    label="Validation",
)
ax_loss.set(xlabel="Step", ylabel="Loss (SMAPE)", ylim=(0, 1))
ax_loss.legend()

ax_scale.plot(history["step"], 10.0 * np.log10(history["scale"]), color=COLORS[0], label="Learned")
ax_scale.axhline(
    -cfg.scenario.measurement.rsrp_noise_mean_db, color=COLORS[1], linestyle="--", label="Expected"
)
ax_scale.axhline(10.0 * np.log10(itu_scale), color=COLORS[2], linestyle=":", label="ITU materials")
ax_scale.set(xlabel="Step", ylabel=r"Power scale $10 \log_{10} \alpha$ (dB)")
ax_scale.legend()
label_panels([ax_loss, ax_scale])
save_figure(fig, "calibration_learning_curves")
plt.show()

**Fig. 2.** Learning curves. (a) Loss of each batch, its moving mean, and the loss of the
validation records. (b) Power scale. The dashed line is the negative of the mean of the
measurement noise. The dotted line is the scale of the baseline with the ITU materials.

## 7. Evaluation on the test records

The error metric is the absolute logarithmic error (ALE) of the paper (Eq. 44). For one
received (record, cell) pair of the test part,

$$\mathrm{ALE} = \left| P_\mathrm{dB} + 10 \log_{10} \alpha - \hat{P}_\mathrm{dB} \right|,$$

where $P_\mathrm{dB}$ is the measured RSRP and $\hat{P}_\mathrm{dB}$ is the simulated RSRP.
Each model uses its own scale $\alpha$. A pair with no simulated path has no ALE. Table 4 gives
their number, because a model that loses many paths can have a small ALE on the other pairs.

In [ ]:
neural_test_dbm = compute_rsrp(scene, cells, positions["test"], cfg.simulation)
models = {
    "ITU materials": (itu_dbm["test"], itu_scale),
    "Neural materials": (neural_test_dbm, result.scale),
}
received = np.isfinite(rsrp_dbm["test"])
ale = {
    name: absolute_log_error(rsrp_dbm["test"], simulated, scale)
    for name, (simulated, scale) in models.items()
}

save_table(
    pd.DataFrame(
        {
            name: {
                "Power scale (dB)": 10.0 * np.log10(models[name][1]),
                "Received pairs": int(received.sum()),
                "Pairs with no simulated path": int((received & np.isnan(models[name][0])).sum()),
                "Mean ALE (dB)": np.nanmean(error),
                "Standard deviation of the ALE (dB)": np.nanstd(error),
                "Median ALE (dB)": np.nanmedian(error),
                "90th percentile of the ALE (dB)": np.nanpercentile(error, 90),
            }
            for name, error in ale.items()
        }
    )
    .style.set_caption("Table 4. Error statistics on the test records.")
    .format(precision=2),
    "calibration_errors",
)

In [ ]:
def record_mean(error):
    "Return the mean ALE of each record. NaN if no pair of the record has an ALE."
    count = np.isfinite(error).sum(axis=1)
    return np.where(count > 0, np.nansum(error, axis=1) / np.maximum(count, 1), np.nan)


record_ale = {name: record_mean(error) for name, error in ale.items()}
ale_max = np.nanpercentile(np.concatenate(list(record_ale.values())), 95)

fig, axes = plt.subplots(1, 3, figsize=(FIGURE_WIDTH_DOUBLE, 2.6), width_ratios=[1, 1.15, 1.15])
for color, (name, error) in zip(COLORS, ale.items(), strict=False):
    axes[0].plot(*ecdf(error[np.isfinite(error)]), color=color, label=name)
axes[0].set(xlabel="ALE (dB)", ylabel="Empirical CDF", ylim=(0, 1))
axes[0].legend(loc="lower right")
for ax, error in zip(axes[1:], record_ale.values(), strict=True):
    ax.imshow(is_building, origin="lower", extent=map_extent, cmap="Greys", alpha=0.3)
    points = ax.scatter(
        *positions["test"][:, :2].T, c=error, s=3, linewidths=0, vmin=0, vmax=ale_max
    )
    format_map(ax)
fig.colorbar(points, ax=axes[1:], label="Mean ALE of the record (dB)", extend="max")
label_panels(axes)
save_figure(fig, "calibration_error")
plt.show()

**Fig. 3.** Error on the test records. (a) Empirical distribution of the ALE of the received
(record, cell) pairs. (b) Mean ALE of each test record with the ITU materials. (c) Mean ALE of
each test record with the neural materials. The color scale stops at the 95th percentile.

## 8. Learned material maps

`NeuralRadioMaterial.values_at` gives the material properties at a set of points. Fig. 4 shows
them at the highest surface of each grid tile: the ground or a roof. The walls of the
buildings are not in these maps.

In [ ]:
columns = height_map.x_min + (np.arange(nx) + 0.5) * height_map.tile_size
rows = height_map.y_min + (np.arange(ny) + 0.5) * height_map.tile_size
grid_x, grid_y = np.meshgrid(columns, rows)
surface = np.column_stack([grid_x.ravel(), grid_y.ravel(), (height_map.z_ground + heights).ravel()])
values = material.values_at(surface)
panels = {
    r"Relative permittivity $\varepsilon_r$": (values.relative_permittivity, True),
    r"Conductivity $\sigma$ (S/m)": (values.conductivity, True),
    "Scattering coefficient $S$": (values.scattering_coefficient, False),
    "XPD coefficient $K_x$": (values.xpd_coefficient, False),
    "Thickness $d$ (m)": (values.thickness, True),
}

fig, axes = plt.subplots(2, 3, figsize=(FIGURE_WIDTH_DOUBLE, 4.6))
for ax, (label, (value, logarithmic)) in zip(axes.flat, panels.items(), strict=False):
    image = ax.imshow(
        value.numpy().reshape(ny, nx),
        origin="lower",
        extent=map_extent,
        norm=LogNorm() if logarithmic else None,
    )
    fig.colorbar(image, ax=ax, label=label)
    format_map(ax)
axes.flat[-1].set_axis_off()
label_panels(axes.flat[: len(panels)])
save_figure(fig, "calibration_material_maps")
plt.show()

**Fig. 4.** Material properties from the calibrated network at the highest surface of each
grid tile. (a) Relative permittivity. (b) Conductivity. (c) Scattering coefficient. (d) XPD
coefficient. (e) Thickness. Panels (a), (b), and (e) have a logarithmic color scale.

## 9. Findings

The cell below prints the results of this run. The notes after it give the limits of the
method. They are valid for each run.

In [ ]:
final = history.iloc[-1]
print(f"Steps: {len(history)}")
print(f"Loss, moving mean of the last {window} steps: {history['loss'].tail(window).mean():.3f}")
if len(validated):
    print(f"Validation loss at the last validation: {validated['validation_loss'].iloc[-1]:.3f}")
print(f"Power scale: {10.0 * np.log10(final['scale']):.2f} dB")
print(f"Expected power scale: {-cfg.scenario.measurement.rsrp_noise_mean_db:.2f} dB")
for name, error in ale.items():
    print(f"Mean ALE on the test records, {name}: {np.nanmean(error):.2f} dB")

- **The baseline is the model that made the data.** The ITU materials are the true materials
  of this synthetic data set. A neural material that gets near the ALE of the baseline has
  found an equivalent environment. It cannot be better than the baseline, apart from the
  effects of the measurement noise.
- **The solution is not unique.** The loss is not convex (arXiv:2311.18558, Section IV-D).
  Different values of $\varepsilon_r$, $\sigma$, and $d$ can give the same reflection and the
  same wall loss. The power scale and the reflectivity of the environment can also compensate
  for each other. A learned property is thus not necessarily the physical property.
- **The RSRP gives less information than a channel impulse response.** The loss has no delay
  spread term, thus it does not see the delay of each path.
- **A random division does not test the generalization in space.** A test record can be near
  a train record. The paper tells that neural materials generalize poorly to areas with no
  measurements.
- **The network starts from random weights.** The first steps use arbitrary materials. A small
  number of steps is not sufficient for convergence.
- **The measurement errors stay in the data.** The reported positions have a GNSS error.
  The RSRP has a resolution of `scenario.measurement.rsrp_resolution_db`. A cell below
  `scenario.measurement.min_rsrp_dbm` is not in a record. The loss does not model these effects.